# LeetCode 1893 — Check if All the Integers in a Range Are Covered bằng Python

Link đề: https://leetcode.com/problems/check-if-all-the-integers-in-a-range-are-covered/

Độ khó: **Easy** · Chủ đề: Array, Hash Table, Prefix Sum

```python
class Solution:
    def isCovered(self, ranges: list[list[int]], left: int, right: int) -> bool:

```

**Mục tiêu học:**

1. Hiểu **đoạn đóng** (inclusive interval) `[start, end]` và khái niệm "một số được phủ".
2. Cách trực giác: thử từng số `x` trong `[left, right]`, hỏi "có đoạn nào chứa `x`?" — dùng `any()`.
3. Cách dùng `set`: đánh dấu mọi điểm được phủ.
4. Kỹ thuật **mảng hiệu (difference array) + prefix sum** — lời giải chính, công cụ cực kỳ hay gặp với bài "cộng trên đoạn".
5. Cách **sắp xếp + quét tham lam**: đẩy dần mốc `left` — dùng được cả khi tọa độ rất lớn.

## 1. Bài toán yêu cầu gì?

- `ranges[i] = [start_i, end_i]` là một đoạn **đóng**: chứa mọi số nguyên `x` với `start_i <= x <= end_i`.
- Cho thêm `left`, `right`. Hỏi: **mọi** số nguyên trong `[left, right]` có được phủ bởi **ít nhất một** đoạn không?

| ranges | left | right | Kết quả | Vì sao |
|---|:-:|:-:|:-:|---|
| `[[1,2],[3,4],[5,6]]` | 2 | 5 | `True` | 2 ∈ [1,2]; 3,4 ∈ [3,4]; 5 ∈ [5,6] |
| `[[1,10],[10,20]]` | 21 | 21 | `False` | 21 nằm ngoài mọi đoạn |
| `[[1,5]]` | 3 | 7 | `False` | 6, 7 không được phủ |

**Ràng buộc:** `1 <= len(ranges) <= 50`, `1 <= start_i <= end_i <= 50`, `1 <= left <= right <= 50`.

Chú ý ví dụ 1: các đoạn `[1,2]` và `[3,4]` **không chồng lên nhau** nhưng vẫn "nối liền" vì đây là số **nguyên** — không có số nguyên nào giữa 2 và 3.

In [1]:
examples = [
    ([[1, 2], [3, 4], [5, 6]], 2, 5, True),
    ([[1, 10], [10, 20]], 21, 21, False),
    ([[1, 5]], 3, 7, False),
]
for ranges, left, right, expected in examples:
    print(f"ranges={ranges}, [left, right]=[{left}, {right}] -> kỳ vọng {expected}")

ranges=[[1, 2], [3, 4], [5, 6]], [left, right]=[2, 5] -> kỳ vọng True
ranges=[[1, 10], [10, 20]], [left, right]=[21, 21] -> kỳ vọng False
ranges=[[1, 5]], [left, right]=[3, 7] -> kỳ vọng False


## 2. Nền tảng Python cần biết

### 2.1. `range(a, b + 1)` — duyệt đoạn **đóng**

`range(a, b)` dừng **trước** `b`. Muốn lấy cả `b` phải viết `range(a, b + 1)`. Đây là nguồn lỗi "lệch một" (off-by-one) số 1 trong bài này.

In [2]:
print(list(range(2, 5)))       # [2, 3, 4]   thiếu 5!
print(list(range(2, 5 + 1)))   # [2, 3, 4, 5]

[2, 3, 4]
[2, 3, 4, 5]


### 2.2. Kiểm tra "x nằm trong đoạn" bằng so sánh chuỗi

Python cho viết `start <= x <= end` thay cho `start <= x and x <= end`.

### 2.3. Unpack (tách) một cặp

`for start, end in ranges:` tự tách mỗi `[start, end]` thành hai biến.

In [3]:
ranges = [[1, 2], [3, 4], [5, 6]]
for start, end in ranges:
    print(f"đoạn [{start}, {end}] chứa 4? {start <= 4 <= end}")

đoạn [1, 2] chứa 4? False
đoạn [3, 4] chứa 4? True
đoạn [5, 6] chứa 4? False


### 2.4. `any()` và `all()`

- `any(điều_kiện for ...)` → `True` nếu **có ít nhất một** phần tử thỏa.
- `all(điều_kiện for ...)` → `True` nếu **mọi** phần tử thỏa.

Cả hai **dừng sớm**: `any` dừng ở phần tử `True` đầu tiên, `all` dừng ở phần tử `False` đầu tiên.

Bài này đọc đúng như vậy: "**mọi** `x` (all) đều có **ít nhất một** đoạn (any) chứa nó".

In [4]:
print(any(s <= 4 <= e for s, e in ranges))    # True
print(any(s <= 7 <= e for s, e in ranges))    # False
print(all(x % 2 == 0 for x in [2, 4, 6]))     # True
print(all(x % 2 == 0 for x in [2, 3, 6]))     # False

True
False
True
False


### 2.5. `set` — tập hợp không trùng lặp, tra cứu nhanh

- `s = set()` tạo tập rỗng; `s.add(x)` thêm; `x in s` kiểm tra trung bình `O(1)`.
- `s.update(iterable)` thêm nhiều phần tử một lúc.

In [5]:
covered = set()
covered.update(range(1, 2 + 1))
covered.update(range(3, 4 + 1))
covered.add(4)                     # thêm trùng: không sao
print(covered, 3 in covered, 5 in covered)

{1, 2, 3, 4} True False


### 2.6. Prefix sum (tổng tiền tố) — nhắc nhanh

Với mảng `a`, `prefix[i] = a[0] + a[1] + ... + a[i]`. Tính dồn trong một vòng: `running += a[i]`.

In [6]:
a = [3, -1, 0, 2, -2]
running = 0
prefix = []
for v in a:
    running += v
    prefix.append(running)
print(prefix)

[3, 2, 2, 4, 2]


## 3. Cách 1 — Thử từng số (trực giác, đúng như hint của đề)

Với mỗi `x` từ `left` đến `right`:

- Nếu **không** đoạn nào chứa `x` → trả `False` ngay.

Duyệt hết mà không bị bắt lỗi → `True`.

In [7]:
def is_covered_brute(ranges, left, right):
    for x in range(left, right + 1):
        found = False
        for start, end in ranges:
            if start <= x <= end:
                found = True
                break
        if not found:
            return False
    return True

for r, l, rr, e in examples:
    print(is_covered_brute(r, l, rr), "| kỳ vọng", e)

True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False


Viết gọn bằng `all` + `any`:

In [8]:
def is_covered_any_all(ranges, left, right):
    return all(any(s <= x <= e for s, e in ranges) for x in range(left, right + 1))

for r, l, rr, e in examples:
    print(is_covered_any_all(r, l, rr), "| kỳ vọng", e)

True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False


### Code nộp LeetCode — Cách 1

In [9]:
class Solution:
    def isCovered(self, ranges: list[list[int]], left: int, right: int) -> bool:
        return all(any(s <= x <= e for s, e in ranges) for x in range(left, right + 1))


print(Solution().isCovered([[1, 2], [3, 4], [5, 6]], 2, 5))  # True

True


**Độ phức tạp Cách 1:** gọi `m = len(ranges)`, `L = right - left + 1`. Thời gian `O(L · m)`, bộ nhớ `O(1)`.

Với ràng buộc ≤ 50 thì tối đa 2.500 phép so sánh — rất nhanh. Nhưng **nút thắt** là: với mỗi `x`, ta lại quét **toàn bộ** danh sách đoạn. Nếu đề mở rộng thành `10^5` đoạn và `10^5` điểm, `O(L·m) = 10^10` là quá chậm. Câu hỏi: có thể "tô màu" các đoạn **một lần**, rồi hỏi từng điểm thật nhanh không?

## 4. Cách 2 — Đánh dấu điểm bằng `set`

Duyệt từng đoạn, thêm mọi điểm của nó vào `covered`. Sau đó hỏi từng `x ∈ [left, right]` có trong `covered` không.

In [10]:
def is_covered_set(ranges, left, right):
    covered = set()
    for start, end in ranges:
        covered.update(range(start, end + 1))
    return all(x in covered for x in range(left, right + 1))

for r, l, rr, e in examples:
    print(is_covered_set(r, l, rr), "| kỳ vọng", e)

True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False


**Độ phức tạp Cách 2:** thời gian `O(tổng độ dài các đoạn + L)` (tệ nhất `O(m · V)` với `V = 50` là miền giá trị), bộ nhớ `O(V)`.

Vẫn còn nút thắt: một đoạn dài `[1, 10^9]` sẽ thêm 10^9 điểm. Ta muốn mỗi đoạn tốn **O(1)** công sức, bất kể dài bao nhiêu.

## 5. Insight — Mảng hiệu (difference array)

Thay vì cộng 1 cho **từng** điểm trong `[s, e]`, ta chỉ đánh dấu **hai mốc**:

- `diff[s] += 1` — "từ `s` trở đi, độ phủ tăng 1".
- `diff[e + 1] -= 1` — "từ `e + 1` trở đi, hết tác dụng".

Sau đó **prefix sum** của `diff` cho ra `cover[x]` = số đoạn chứa `x`.

Ví dụ `ranges = [[1,2],[3,4],[5,6]]`, chỉ số 0..7:

| x | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| `diff` | 0 | +1 | 0 | +1−1=0 | 0 | +1−1=0 | 0 | −1 |
| `cover` (prefix) | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |

Đoạn `[3,4]`: `+1` tại 3 và đoạn `[1,2]`: `−1` tại 3 triệt tiêu nhau — độ phủ vẫn liên tục bằng 1.

In [11]:
ranges = [[1, 2], [3, 4], [5, 6]]
diff = [0] * 8
for s, e in ranges:
    diff[s] += 1
    diff[e + 1] -= 1
print("diff :", diff)

cover = []
running = 0
for d in diff:
    running += d
    cover.append(running)
print("cover:", cover)

diff : [0, 1, 0, 0, 0, 0, 0, -1]
cover: [0, 1, 1, 1, 1, 1, 1, 0]


## 6. Lời giải chính — mảng hiệu, giảng từng đoạn rồi ghép

**Đoạn 1 — kích thước mảng.** Giá trị lớn nhất là 50, ta cần cả chỉ số `e + 1 = 51`, nên dùng mảng 52 phần tử.

**Đoạn 2 — đánh dấu hai mốc cho mỗi đoạn.**

**Đoạn 3 — quét prefix sum**, đến khi gặp `x ∈ [left, right]` thì kiểm tra `running > 0`. Không cần lưu cả mảng `cover`; một biến `running` là đủ.

In [12]:
def is_covered_diff(ranges, left, right):
    MAX = 50
    diff = [0] * (MAX + 2)          # chỉ số 0..51
    for s, e in ranges:
        diff[s] += 1
        diff[e + 1] -= 1

    running = 0
    for x in range(1, right + 1):
        running += diff[x]          # running = số đoạn chứa x
        if x >= left and running <= 0:
            return False
    return True

for r, l, rr, e in examples:
    print(is_covered_diff(r, l, rr), "| kỳ vọng", e)

True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False


### Vì sao vòng quét bắt đầu từ 1 chứ không phải `left`?

`running` là tổng **từ đầu mảng**. Nếu bắt đầu ở `left`, ta bỏ qua các `+1` của những đoạn bắt đầu trước `left` (ví dụ đoạn `[1, 10]` với `left = 5`), và `running` sẽ sai. Phải cộng dồn từ đầu, chỉ **kiểm tra** khi `x >= left`.

In [13]:
def wrong_start_at_left(ranges, left, right):
    diff = [0] * 52
    for s, e in ranges:
        diff[s] += 1
        diff[e + 1] -= 1
    running = 0
    for x in range(left, right + 1):   # SAI: bỏ qua phần trước left
        running += diff[x]
        if running <= 0:
            return False
    return True

print("Sai :", wrong_start_at_left([[1, 10]], 5, 7))
print("Đúng:", is_covered_diff([[1, 10]], 5, 7))

Sai : False
Đúng: True


## 7. TRACE từng bước (mảng hiệu)

Ví dụ `ranges = [[1,10],[10,20]]`, `left = right = 21` (ví dụ 2), và một ví dụ có lỗ hổng `ranges = [[1,3],[5,6]]`, `left = 2`, `right = 6`.

Hàm trace in: `diff` sau khi đánh dấu, rồi từng `x`, `diff[x]`, `running`, và quyết định.

In [14]:
def trace_is_covered_diff(ranges, left, right):
    diff = [0] * 52
    for s, e in ranges:
        diff[s] += 1
        diff[e + 1] -= 1
        print(f"đoạn [{s},{e}]: diff[{s}] += 1, diff[{e + 1}] -= 1")
    nz = {i: v for i, v in enumerate(diff) if v != 0}
    print("diff (chỉ ô khác 0):", nz)
    running = 0
    for x in range(1, right + 1):
        running += diff[x]
        if x >= left:
            status = "phủ ✓" if running > 0 else "KHÔNG phủ ✗ -> return False"
            print(f"  x={x:2}: diff[x]={diff[x]:+d}, running={running} -> {status}")
            if running <= 0:
                return False
    print("  Mọi x trong đoạn đều được phủ -> True")
    return True

trace_is_covered_diff([[1, 3], [5, 6]], 2, 6)

đoạn [1,3]: diff[1] += 1, diff[4] -= 1
đoạn [5,6]: diff[5] += 1, diff[7] -= 1
diff (chỉ ô khác 0): {1: 1, 4: -1, 5: 1, 7: -1}
  x= 2: diff[x]=+0, running=1 -> phủ ✓
  x= 3: diff[x]=+0, running=1 -> phủ ✓
  x= 4: diff[x]=-1, running=0 -> KHÔNG phủ ✗ -> return False


False

In [15]:
trace_is_covered_diff([[1, 10], [10, 20]], 21, 21)

đoạn [1,10]: diff[1] += 1, diff[11] -= 1
đoạn [10,20]: diff[10] += 1, diff[21] -= 1
diff (chỉ ô khác 0): {1: 1, 10: 1, 11: -1, 21: -1}
  x=21: diff[x]=-1, running=0 -> KHÔNG phủ ✗ -> return False


False

Quan sát ở trace đầu: tại `x = 4`, `diff[4] = -1` (đoạn `[1,3]` kết thúc), `running` về 0 → lỗ hổng. Ở trace thứ hai, `x = 21` có `diff[21] = -1` (đoạn `[10,20]` kết thúc) đưa `running` về 0.

## 8. Cách 4 — Sắp xếp + quét tham lam (đẩy mốc `left`)

Ý tưởng: ta cần phủ `left` trước tiên. Sắp xếp các đoạn theo `start`. Duyệt lần lượt:

- Nếu `start <= left <= end` → đoạn này phủ `left`, và cả mọi số tới `end`. Đẩy `left = end + 1`.
- Khi `left > right` → đã phủ xong.

Vì sao phải sắp xếp? Nếu một đoạn có `start` nhỏ xuất hiện **sau**, ta sẽ bỏ lỡ nó lúc cần. Sắp xếp đảm bảo khi `left` tăng, mọi đoạn có thể phủ nó đã được xét hoặc sắp được xét.

`sorted(ranges)` sắp các list theo phần tử đầu, hòa thì theo phần tử thứ hai.

In [16]:
print(sorted([[5, 6], [1, 2], [3, 4], [1, 1]]))

[[1, 1], [1, 2], [3, 4], [5, 6]]


In [17]:
def is_covered_greedy(ranges, left, right):
    for s, e in sorted(ranges):
        if s <= left <= e:
            left = e + 1
        if left > right:
            return True
    return left > right

for r, l, rr, e in examples:
    print(is_covered_greedy(r, l, rr), "| kỳ vọng", e)

True | kỳ vọng True
False | kỳ vọng False
False | kỳ vọng False


In [18]:
def trace_greedy(ranges, left, right):
    print("sau sắp xếp:", sorted(ranges))
    for s, e in sorted(ranges):
        if s <= left <= e:
            print(f"  [{s},{e}] phủ left={left} -> left = {e + 1}")
            left = e + 1
        else:
            print(f"  [{s},{e}] không chứa left={left} -> bỏ qua")
        if left > right:
            print("  left > right -> True")
            return True
    print(f"  hết đoạn, left={left} <= right={right} -> False")
    return False

trace_greedy([[5, 6], [3, 4], [1, 2]], 2, 5)
print()
trace_greedy([[1, 3], [5, 6]], 2, 6)

sau sắp xếp: [[1, 2], [3, 4], [5, 6]]
  [1,2] phủ left=2 -> left = 3
  [3,4] phủ left=3 -> left = 5
  [5,6] phủ left=5 -> left = 7
  left > right -> True

sau sắp xếp: [[1, 3], [5, 6]]
  [1,3] phủ left=2 -> left = 4
  [5,6] không chứa left=4 -> bỏ qua
  hết đoạn, left=4 <= right=6 -> False


False

**Độ phức tạp Cách 4:** `O(m log m)` cho sắp xếp + `O(m)` quét; bộ nhớ `O(m)` cho bản sắp xếp. **Không phụ thuộc** độ lớn tọa độ — dùng được khi giá trị tới `10^9`.

## 9. Code nộp LeetCode — lời giải chính (mảng hiệu)

In [19]:
class Solution:
    def isCovered(self, ranges: list[list[int]], left: int, right: int) -> bool:
        diff = [0] * 52
        for s, e in ranges:
            diff[s] += 1
            diff[e + 1] -= 1
        running = 0
        for x in range(1, right + 1):
            running += diff[x]
            if x >= left and running <= 0:
                return False
        return True


sol = Solution()
for r, l, rr, e in examples:
    assert sol.isCovered(r, l, rr) == e
print("Solution (mảng hiệu) PASS")

Solution (mảng hiệu) PASS


Bản tham lam để submit (cũng Accepted):

In [20]:
class SolutionGreedy:
    def isCovered(self, ranges: list[list[int]], left: int, right: int) -> bool:
        for s, e in sorted(ranges):
            if s <= left <= e:
                left = e + 1
        return left > right


for r, l, rr, e in examples:
    assert SolutionGreedy().isCovered(r, l, rr) == e
print("SolutionGreedy PASS")

SolutionGreedy PASS


Ở bản gọn này ta không `return` sớm: khi `left > right` rồi, điều kiện `s <= left <= e` có thể vẫn đúng và đẩy `left` xa hơn — không sao, `left > right` vẫn giữ.

## 10. Vì sao đúng?

**Mảng hiệu.** Invariant của vòng quét: sau khi cộng `diff[1..x]`, `running` bằng số đoạn `[s, e]` thỏa `s <= x` trừ số đoạn thỏa `e + 1 <= x`, tức là **số đoạn có `s <= x <= e`** — chính xác số đoạn chứa `x`. Vậy `x` được phủ ⇔ `running > 0`. Ta kiểm tra điều này cho mọi `x ∈ [left, right]` nên kết quả đúng.

**Tham lam.** Invariant: mọi số nguyên trong `[left_ban_đầu, left - 1]` đã được phủ. Khi xét đoạn `[s, e]` (theo `s` tăng dần):

- nếu `s <= left <= e`, cả `[left, e]` được phủ → đẩy `left = e + 1` vẫn giữ invariant;
- nếu `left < s`: mọi đoạn còn lại có `start >= s > left` nên **không bao giờ** phủ được `left` → kết quả cuối sẽ là `False` (vòng lặp chỉ bỏ qua, `left` không đổi);
- nếu `e < left`: đoạn nằm trọn trong phần đã phủ, bỏ qua an toàn.

Kết thúc: `left > right` ⇔ toàn bộ `[left, right]` đã phủ.

## 11. Độ phức tạp và bảng so sánh

`m = len(ranges)`, `L = right - left + 1`, `V` = giá trị lớn nhất (50).

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Khi tọa độ tới 10^9? |
|---|---|:---:|:---:|---|
| 1. Thử từng số | `all(any(...))` | `O(L·m)` | `O(1)` | chậm nếu `L` lớn |
| 2. `set` đánh dấu | thêm mọi điểm | `O(Σ độ dài + L)` | `O(V)` | không dùng được |
| 3. Mảng hiệu + prefix | `+1` ở `s`, `−1` ở `e+1` | `O(m + V)` | `O(V)` | cần nén tọa độ |
| 4. Sắp xếp + tham lam | đẩy `left` | `O(m log m)` | `O(m)` | dùng tốt |

Với ràng buộc của đề (mọi thứ ≤ 50) cả 4 cách đều chạy tức thì. Mảng hiệu là kỹ thuật đáng học nhất vì tái sử dụng rất nhiều (Car Pooling, Corporate Flight Bookings, Range Addition...).

## 12. Bộ test kiểm tra

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:30} PASS")

solvers = [is_covered_brute, is_covered_any_all, is_covered_set, is_covered_diff, is_covered_greedy]

tests = [
    ("ví dụ 1", [[1, 2], [3, 4], [5, 6]], 2, 5, True),
    ("ví dụ 2", [[1, 10], [10, 20]], 21, 21, False),
    ("phủ một phần", [[1, 5]], 3, 7, False),
    ("một điểm, được phủ", [[7, 7]], 7, 7, True),
    ("lỗ ở giữa", [[1, 3], [5, 6]], 2, 6, False),
    ("đoạn lồng nhau", [[1, 50], [2, 3]], 1, 50, True),
    ("chưa sắp xếp", [[5, 6], [3, 4], [1, 2]], 1, 6, True),
    ("thiếu đầu left", [[2, 50]], 1, 50, False),
    ("thiếu cuối right", [[1, 49]], 1, 50, False),
    ("đoạn nằm ngoài", [[30, 40]], 1, 10, False),
]
for name, r, l, rr, e in tests:
    for f in solvers:
        assert f(r, l, rr) == e, (name, f.__name__)
    check(name + " (5 cách)", is_covered_diff(r, l, rr), e)

ví dụ 1 (5 cách)               PASS
ví dụ 2 (5 cách)               PASS
phủ một phần (5 cách)          PASS
một điểm, được phủ (5 cách)    PASS
lỗ ở giữa (5 cách)             PASS
đoạn lồng nhau (5 cách)        PASS
chưa sắp xếp (5 cách)          PASS
thiếu đầu left (5 cách)        PASS
thiếu cuối right (5 cách)      PASS
đoạn nằm ngoài (5 cách)        PASS


In [22]:
import random

random.seed(1893)
mismatch = 0
for _ in range(3000):
    k = random.randint(1, 6)
    r = []
    for _ in range(k):
        s = random.randint(1, 50)
        e = random.randint(s, min(50, s + random.randint(0, 15)))
        r.append([s, e])
    l = random.randint(1, 50)
    rr = random.randint(l, min(50, l + 20))
    results = {f(r, l, rr) for f in solvers}
    if len(results) != 1:
        mismatch += 1
print("Số ca các cách lệch nhau:", mismatch)
assert mismatch == 0

Số ca các cách lệch nhau: 0


## 13. Lỗi người mới hay gặp

### Lỗi 1: `range(left, right)` bỏ mất `right`

```python
for x in range(left, right):   # SAI: không kiểm tra x = right
```

In [23]:
def wrong_range(ranges, left, right):
    return all(any(s <= x <= e for s, e in ranges) for x in range(left, right))

print("Sai :", wrong_range([[1, 49]], 1, 50), "| Đúng:", is_covered_any_all([[1, 49]], 1, 50))

Sai : True | Đúng: False


### Lỗi 2: Đặt `-1` tại `e` thay vì `e + 1`

```python
diff[e] -= 1     # SAI: làm điểm e mất độ phủ
```

Đoạn đóng nghĩa là `e` **vẫn** được phủ; tác dụng chỉ hết từ `e + 1`.

In [24]:
def wrong_diff_end(ranges, left, right):
    diff = [0] * 52
    for s, e in ranges:
        diff[s] += 1
        diff[e] -= 1          # SAI
    running = 0
    for x in range(1, right + 1):
        running += diff[x]
        if x >= left and running <= 0:
            return False
    return True

print("Sai :", wrong_diff_end([[1, 2], [3, 4], [5, 6]], 2, 5), "| Đúng:", is_covered_diff([[1, 2], [3, 4], [5, 6]], 2, 5))

Sai : False | Đúng: True


### Lỗi 3: Mảng hiệu quá nhỏ

`diff = [0] * 51` → `diff[50 + 1]` gây `IndexError` khi có đoạn kết thúc ở 50. Phải có chỗ cho chỉ số `e + 1 = 51`.

### Lỗi 4: Tham lam mà quên sắp xếp

In [25]:
def wrong_greedy_no_sort(ranges, left, right):
    for s, e in ranges:          # SAI: không sorted
        if s <= left <= e:
            left = e + 1
    return left > right

r = [[3, 4], [1, 2]]
print("Sai :", wrong_greedy_no_sort(r, 1, 4), "| Đúng:", is_covered_greedy(r, 1, 4))

Sai : False | Đúng: True


Vì sao sai: `[3,4]` được xét khi `left = 1` → bỏ qua; sau đó `[1,2]` đẩy `left = 3`, nhưng không còn đoạn nào để phủ 3.

### Lỗi 5: Kiểm tra "mỗi đoạn nằm trong [left, right]"

Đọc nhầm đề thành "mỗi đoạn có nằm trong `[left, right]` không". Đề hỏi chiều ngược lại: mỗi **điểm** trong `[left, right]` có thuộc **một** đoạn nào không.

## 14. Mẹo nhận dạng pattern và biến thể

- "Nhiều đoạn, cộng/đánh dấu trên cả đoạn, rồi hỏi từng điểm" → **mảng hiệu + prefix sum**.
- "Các đoạn có phủ kín một khoảng không / gộp đoạn" → **sắp xếp theo start + quét** (giống 56 Merge Intervals).
- Miền giá trị nhỏ (≤ 50, ≤ 1000) → mảng đếm trực tiếp là đủ.

Bài liên quan: 56 (Merge Intervals), 1094 (Car Pooling), 1109 (Corporate Flight Bookings), 370 (Range Addition), 252 (Meeting Rooms).

### Biến thể 1: mỗi điểm phải được phủ **ít nhất k lần**

Mảng hiệu xử lý ngay: thay `running <= 0` bằng `running < k`.

In [26]:
def is_covered_k(ranges, left, right, k):
    diff = [0] * 52
    for s, e in ranges:
        diff[s] += 1
        diff[e + 1] -= 1
    running = 0
    for x in range(1, right + 1):
        running += diff[x]
        if x >= left and running < k:
            return False
    return True

print(is_covered_k([[1, 5], [3, 8]], 3, 5, 2))   # 3..5 được phủ 2 lần -> True
print(is_covered_k([[1, 5], [3, 8]], 2, 5, 2))   # 2 chỉ phủ 1 lần -> False

True
False


### Biến thể 2: liệt kê các số **chưa** được phủ

In [27]:
def uncovered_points(ranges, left, right):
    diff = [0] * 52
    for s, e in ranges:
        diff[s] += 1
        diff[e + 1] -= 1
    out, running = [], 0
    for x in range(1, right + 1):
        running += diff[x]
        if x >= left and running == 0:
            out.append(x)
    return out

print(uncovered_points([[1, 3], [5, 6], [9, 9]], 1, 10))

[4, 7, 8, 10]


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Tính tay mảng hiệu

Với `ranges = [[2,4],[3,6]]`, viết `diff[0..7]` và `cover[0..7]` (prefix sum) vào hai list.

In [28]:
ex1_diff = None    # TODO: 8 số
ex1_cover = None   # TODO: 8 số

### Bài 2 — Điền chỗ trống: lời giải `set`

In [29]:
def ex2_is_covered(ranges, left, right):
    covered = set()
    for start, end in ranges:
        # TODO: thêm mọi điểm của đoạn vào covered
        pass
    # TODO: return ...
    pass

# assert ex2_is_covered([[1, 2], [3, 4], [5, 6]], 2, 5) == True
# assert ex2_is_covered([[1, 10], [10, 20]], 21, 21) == False

### Bài 3 — Tự viết lời giải mảng hiệu (không nhìn code phía trên)

In [30]:
def ex3_is_covered(ranges, left, right):
    # TODO
    pass

# for r, l, rr, e in examples: assert ex3_is_covered(r, l, rr) == e

### Bài 4 — Biến thể: điểm được phủ nhiều nhất

Trả về `(x, số_lần)` với `x` nhỏ nhất có số đoạn phủ lớn nhất (giá trị trong `1..50`).

In [31]:
def most_covered_point(ranges):
    # TODO: mảng hiệu + prefix sum, theo dõi max
    pass

# assert most_covered_point([[1, 4], [3, 6], [4, 9]]) == (4, 3)

### Bài 5 — Tham lam với tọa độ lớn

Viết `is_covered_big(ranges, left, right)` đúng với tọa độ tới `10^9` (không được tạo mảng kích thước tọa độ).

In [32]:
def is_covered_big(ranges, left, right):
    # TODO
    pass

# assert is_covered_big([[1, 10**9]], 5, 10**9) == True
# assert is_covered_big([[1, 10], [12, 10**9]], 5, 100) == False

### Bài 6 — Mini challenge: số đoạn ít nhất để phủ kín

Trả về số đoạn **ít nhất** (chọn từ `ranges`) để phủ kín `[left, right]`, hoặc `-1` nếu không thể (giống 1326/1024 Video Stitching ở dạng số nguyên).

In [33]:
def min_intervals_to_cover(ranges, left, right):
    # TODO: sắp xếp; mỗi bước chọn đoạn có start <= left với end lớn nhất
    pass

# assert min_intervals_to_cover([[1, 3], [2, 6], [4, 8], [7, 10]], 1, 10) == 3
# assert min_intervals_to_cover([[1, 3], [5, 6]], 1, 6) == -1

## 16. Gợi ý 3 mức

**Bài 1.** (1) `+1` tại start, `−1` tại end+1. (2) `[2,4]` → `diff[2]+=1, diff[5]-=1`. (3) `[3,6]` → `diff[3]+=1, diff[7]-=1`; prefix: 0,0,1,2,2,1,1,0.

**Bài 2.** (1) Đoạn đóng. (2) `covered.update(range(start, end + 1))`. (3) `return all(x in covered for x in range(left, right + 1))`.

**Bài 3.** (1) Mảng 52 phần tử. (2) Quét từ 1, không từ `left`. (3) Trả `False` khi `x >= left and running <= 0`.

**Bài 4.** (1) Tính `running` như lời giải chính. (2) Giữ `best_x`, `best_cnt`. (3) Cập nhật khi `running > best_cnt` (dấu `>` nghiêm ngặt để giữ `x` nhỏ nhất).

**Bài 5.** (1) Dùng Cách 4. (2) `sorted(ranges)`. (3) Đẩy `left = e + 1` khi `s <= left <= e`.

**Bài 6.** (1) Tham lam: trong các đoạn bắt đầu `<= left`, chọn đoạn vươn xa nhất. (2) Dùng con trỏ `i` trên danh sách đã sắp. (3) Nếu không đoạn nào vươn tới `>= left` → `-1`; mỗi lần chọn: `count += 1; left = best_end + 1`.

## 17. Đáp án tham khảo (chỉ xem sau khi tự làm)

In [34]:
ans_ex1_diff = [0, 0, 1, 1, 0, -1, 0, -1]
ans_ex1_cover = [0, 0, 1, 2, 2, 1, 1, 0]


def ans_ex2_is_covered(ranges, left, right):
    covered = set()
    for start, end in ranges:
        covered.update(range(start, end + 1))
    return all(x in covered for x in range(left, right + 1))


def ans_ex3_is_covered(ranges, left, right):
    diff = [0] * 52
    for s, e in ranges:
        diff[s] += 1
        diff[e + 1] -= 1
    running = 0
    for x in range(1, right + 1):
        running += diff[x]
        if x >= left and running <= 0:
            return False
    return True


def ans_most_covered_point(ranges):
    diff = [0] * 52
    for s, e in ranges:
        diff[s] += 1
        diff[e + 1] -= 1
    best_x, best_cnt, running = 1, 0, 0
    for x in range(1, 51):
        running += diff[x]
        if running > best_cnt:
            best_x, best_cnt = x, running
    return (best_x, best_cnt)


def ans_is_covered_big(ranges, left, right):
    for s, e in sorted(ranges):
        if s <= left <= e:
            left = e + 1
    return left > right


def ans_min_intervals_to_cover(ranges, left, right):
    rs = sorted(ranges)
    i, count = 0, 0
    while left <= right:
        best_end = left - 1
        while i < len(rs) and rs[i][0] <= left:
            best_end = max(best_end, rs[i][1])
            i += 1
        if best_end < left:
            return -1
        count += 1
        left = best_end + 1
    return count


# kiểm tra bài 1 bằng code
d = [0] * 8
for s, e in [[2, 4], [3, 6]]:
    d[s] += 1; d[e + 1] -= 1
pre, run = [], 0
for v in d:
    run += v; pre.append(run)
assert ans_ex1_diff == d and ans_ex1_cover == pre

for r, l, rr, e in examples:
    assert ans_ex2_is_covered(r, l, rr) == e
    assert ans_ex3_is_covered(r, l, rr) == e
assert ans_most_covered_point([[1, 4], [3, 6], [4, 9]]) == (4, 3)
assert ans_is_covered_big([[1, 10**9]], 5, 10**9) is True
assert ans_is_covered_big([[1, 10], [12, 10**9]], 5, 100) is False
assert ans_min_intervals_to_cover([[1, 3], [2, 6], [4, 8], [7, 10]], 1, 10) == 3
assert ans_min_intervals_to_cover([[1, 3], [5, 6]], 1, 6) == -1
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def isCovered(self, ranges, left, right):
        diff = [0] * 52
        for s, e in ranges:
            diff[s] += 1          # bắt đầu phủ
            diff[e + 1] -= 1      # hết phủ SAU e
        running = 0
        for x in range(1, right + 1):
            running += diff[x]    # số đoạn chứa x
            if x >= left and running <= 0:
                return False
        return True
```

**Khung tư duy mảng hiệu:** "cộng `v` cho cả đoạn `[s, e]`" = `diff[s] += v; diff[e+1] -= v`, rồi prefix sum.

**Checklist trước khi submit:**

- [ ] Đoạn **đóng**: `range(a, b + 1)`, `diff[e + 1]`?
- [ ] Mảng hiệu đủ chỗ cho chỉ số `max + 1`?
- [ ] Prefix sum cộng **từ đầu**, chỉ kiểm tra khi `x >= left`?
- [ ] Nếu dùng tham lam: đã `sorted(ranges)` chưa?
- [ ] Tên method `isCovered`, trả `bool`?